In [1]:
import torch
import torchvision
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf

%config InlineBackend.figure_format = 'svg' 
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

    except RuntimeError as e:
        print(e)

In [9]:

ratings_data = pd.read_csv('C:\\College work\\Advanced ML\\exp 5\\ratings.csv')
movie_names_data = pd.read_csv('C:\\College work\\Advanced ML\\exp 5\\movies.csv')
print("Data loaded successfully!")
print(f"Ratings shape: {ratings_data.shape}")
print(f"Movies shape: {movie_names_data.shape}")


Data loaded successfully!
Ratings shape: (100836, 4)
Movies shape: (9742, 3)


In [11]:
n_movies = len(movie_names_data)
n_user = len(ratings_data['userId'].unique())
print(f"Number of movies: {n_movies}")
print(f"Number of users: {n_user}")

Number of movies: 9742
Number of users: 610


In [12]:
ratings_data = pd.merge(ratings_data, movie_names_data, on='movieId', how='inner')
ratings_data.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [13]:
from sklearn.preprocessing import LabelEncoder
import random
Y = ratings_data.rating
user_enc = LabelEncoder()
movie_enc = LabelEncoder()
X = np.array([user_enc.fit_transform(ratings_data.userId),
              movie_enc.fit_transform(ratings_data.title)]).T

In [14]:
print("Y ", Y[:5])
print("X ", X[:5])

Y  0    4.0
1    4.0
2    4.0
3    5.0
4    5.0
Name: rating, dtype: float64
X  [[   0 8871]
 [   0 3661]
 [   0 3845]
 [   0 7523]
 [   0 9119]]


In [18]:
user_enc.classes_[5], movie_enc.classes_[3661]

(np.int64(6), 'Grumpier Old Men (1995)')

In [19]:
for x, y in zip(X[:10], Y[:10]):
    print(f"X: {list(x)}, Y: {y}")

X: [np.int64(0), np.int64(8871)], Y: 4.0
X: [np.int64(0), np.int64(3661)], Y: 4.0
X: [np.int64(0), np.int64(3845)], Y: 4.0
X: [np.int64(0), np.int64(7523)], Y: 5.0
X: [np.int64(0), np.int64(9119)], Y: 5.0
X: [np.int64(0), np.int64(3252)], Y: 3.0
X: [np.int64(0), np.int64(1284)], Y: 5.0
X: [np.int64(0), np.int64(1337)], Y: 4.0
X: [np.int64(0), np.int64(7180)], Y: 5.0
X: [np.int64(0), np.int64(1535)], Y: 5.0


In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)

In [22]:
num_users = len(X)
num_movies = len(X)

In [23]:
from keras.layers import Input, Embedding, Flatten, Dot, Dense, Activation, Dropout
from keras.models import Model

def build_model():
    movie_input = Input(shape=[1], name="Book-Input")
    movie_embedding = Embedding(n_movies+1, 15, name="Book-Embedding")(movie_input)
    movie_vec = Flatten(name="Flatten-Books")(movie_embedding)

    user_input = Input(shape=[1], name="User-Input")
    user_embedding = Embedding(n_user+1, 15, name="User-Embedding")(user_input)
    user_vec = Flatten(name="Flatten-Users")(user_embedding)
    
    prod = Dot(name="Dot-Product", axes=1)([user_vec, movie_vec])
    
    prod = Dense(32)(prod)
    prod = Activation('relu')(prod)
    prod = Dropout(0.5)(prod)

    prod = Dense(16)(prod)
    prod = Activation('relu')(prod)
    prod = Dropout(0.5)(prod)
    prod = Dense(1)(prod)


    model = Model([user_input, movie_input], prod)
    model.compile('adam', 'mean_squared_error', metrics=['accuracy'])

    return model


model = build_model()

In [25]:
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='./checkpoint.weights.h5',
    save_weights_only=True,
    monitor='val_loss',
    mode='min',
    save_best_only=True,
    verbose=1)

history = model.fit([X_train[:, 0], X_train[:, 1]], Y_train, 
            epochs=15, 
            verbose=1,
            batch_size=64, 
            validation_data=([X_test[:, 0], X_test[:, 1]], Y_test), 
            callbacks=[model_checkpoint_callback])

Epoch 1/15
1244/1261 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0218 - loss: 5.2230
Epoch 1: val_loss improved from None to 1.21167, saving model to ./checkpoint.weights.h5

Epoch 1: finished saving model to ./checkpoint.weights.h5
1261/1261 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.0264 - loss: 3.2346 - val_accuracy: 0.0280 - val_loss: 1.2117
Epoch 2/15
1244/1261 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0275 - loss: 1.9936
Epoch 2: val_loss improved from 1.21167 to 1.16842, saving model to ./checkpoint.weights.h5

Epoch 2: finished saving model to ./checkpoint.weights.h5
1261/1261 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.0279 - loss: 1.8915 - val_accuracy: 0.0280 - val_loss: 1.1684
Epoch 3/15
1260/1261 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0289 - loss: 1.5985
Epoch 3: val_loss improved from 1.16842 to 1.04168, saving model to ./checkpoint.weights.h5

Epoch 3: finished saving model to ./checkpoint.weights.h5
1261/1261 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - acc

In [26]:
X_test[:5], Y_test[:5]

(array([[ 275, 4337],
        [ 598, 7425],
        [ 482,  334],
        [ 201, 3548],
        [ 273, 3540]]),
 41008    5.0
 94274    2.5
 77380    2.5
 29744    3.0
 40462    4.0
 Name: rating, dtype: float64)

In [27]:
predictions = model.predict([X_test[:5, 0], X_test[:5, 1]])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step


In [28]:
print(predictions,"\n\n", Y_test[:5].values)


[[4.160441 ]
 [3.5902853]
 [2.8637896]
 [4.144992 ]
 [3.1654177]] 

 [5.  2.5 2.5 3.  4. ]


In [29]:
movie_enc.classes_[4]

"'Til There Was You (1997)"

In [31]:
def extract_true_ratings(user_id, X_test):
    
    true_ratings = list()
    for x, y in X_test:
        if x == user_id:
            rating = ratings_data[(ratings_data['userId'] == user_enc.classes_[user_id]) \
                & (ratings_data['title'] == movie_enc.classes_[y])]['rating'].values[0]
            true_ratings.append(rating)

    return true_ratings

In [ ]:
test_user_id = 0  # or any user id between 0 and 609
extract_true_ratings(test_user_id, X_test)

NameError: name 'test_user_id' is not defined